# Unit 7: Speech-to-Speech Translation - Hands-on Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 7](https://huggingface.co/learn/audio-course/chapter7)

## Certificate Exercise Requirements

- Duplicate the [template Space](https://huggingface.co/spaces/course-demos/speech-to-speech-translation)
- Update it to translate speech to a **non-English** target language
- Deploy as a **public** Gradio demo on HF Spaces
- Submit to [assessment Space](https://huggingface.co/spaces/huggingface-course/audio-course-u7-assessment)

## Status: TODO

## Architecture Overview

The speech-to-speech translation pipeline is cascaded:

1. **ASR/Translation**: Speech (language X) -> Text (language Y) using Whisper
2. **TTS**: Text (language Y) -> Speech (language Y) using SpeechT5 or MMS

```
Input Speech (any language)
    -> Whisper (translate to target language text)
    -> TTS model (synthesize target language speech)
-> Output Speech (target language)
```

## Setup

In [ ]:
!pip install -q transformers datasets gradio torch soundfile numpy

## 1. Translation Component (Whisper)

Whisper can translate from any of its supported languages to English.
For non-English targets, we use a two-step approach:
1. Transcribe source speech to source text
2. Translate source text to target text (or use Whisper's direct translation)

In [ ]:
from transformers import pipeline
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Step 1: Whisper transcribes/translates speech to English text
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
)

# Step 2: Translate English text to Dutch
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-nl",
    device=device,
)

def translate(audio):
    """Translate speech from any language to Dutch text."""
    # Whisper translates any language to English
    outputs = asr_pipe(
        audio,
        max_new_tokens=256,
        generate_kwargs={"task": "translate"},
    )
    english_text = outputs["text"]

    # Translate English to Dutch
    translated = translator(english_text, max_length=512)
    return translated[0]["translation_text"]

## 2. TTS Component

Options for non-English TTS:
- **Your fine-tuned SpeechT5** from Unit 6
- **MMS TTS** (supports 1100+ languages)
- **Pre-trained SpeechT5**: `sanchit-gandhi/speecht5_tts_vox_nl` (Dutch)

In [ ]:
from transformers import VitsModel, VitsTokenizer
import numpy as np

# MMS TTS for Dutch - better quality than SpeechT5 for non-English
# Alternative: use your fine-tuned SpeechT5 from Unit 6
tts_model = VitsModel.from_pretrained("facebook/mms-tts-nld")
tts_tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-nld")

def synthesise(text):
    """Convert Dutch text to speech using MMS TTS."""
    inputs = tts_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = tts_model(inputs["input_ids"])
    return outputs.waveform[0].numpy()

## 3. Full Pipeline

In [ ]:
import numpy as np

target_dtype = np.int16
max_range = np.iinfo(target_dtype).max

def speech_to_speech_translation(audio):
    """Full speech-to-speech translation pipeline: any language -> Dutch."""
    translated_text = translate(audio)
    print(f"Translated text: {translated_text}")

    synthesised_speech = synthesise(translated_text)

    # Convert to int16 for audio output
    synthesised_speech = (synthesised_speech * max_range).astype(np.int16)

    return 16000, synthesised_speech

## 4. Test Locally

In [ ]:
import IPython.display as ipd

# Test with a sample
from datasets import load_dataset

test_data = load_dataset("PolyAI/minds14", name="en-US", split="train[0:1]", trust_remote_code=True)
test_audio = test_data[0]["audio"]["array"]

print("Input audio:")
ipd.Audio(test_audio, rate=test_data[0]["audio"]["sampling_rate"])

In [ ]:
sr, output_audio = speech_to_speech_translation(test_audio)
print("\nOutput audio:")
ipd.Audio(output_audio, rate=sr)

## 5. Build Gradio Demo

This is the code to deploy as a HuggingFace Space.

In [ ]:
import gradio as gr

demo = gr.Blocks()

with demo:
    gr.Markdown("# Speech-to-Speech Translation")
    gr.Markdown("Translate speech from any language to the target language.")
    
    with gr.Tab("Microphone"):
        mic_input = gr.Audio(sources="microphone", type="numpy")
        mic_output = gr.Audio(label="Translated Speech", type="numpy")
        mic_button = gr.Button("Translate")
        mic_button.click(
            speech_to_speech_translation,
            inputs=mic_input,
            outputs=mic_output,
        )
    
    with gr.Tab("Upload File"):
        file_input = gr.Audio(sources="upload", type="numpy")
        file_output = gr.Audio(label="Translated Speech", type="numpy")
        file_button = gr.Button("Translate")
        file_button.click(
            speech_to_speech_translation,
            inputs=file_input,
            outputs=file_output,
        )

demo.launch()

## 6. Deploy to HuggingFace Spaces

### Steps:

1. **Duplicate** the [template Space](https://huggingface.co/spaces/course-demos/speech-to-speech-translation?duplicate=true)
2. Replace `app.py` with the code below (or adapt the translate/synthesise functions)
3. Set `requirements.txt` to:
   ```
   transformers
   torch
   gradio
   sentencepiece
   ```
4. Ensure the Space visibility is **public**
5. Test by recording/uploading audio
6. Submit your Space ID to the [assessment page](https://huggingface.co/spaces/huggingface-course/audio-course-u7-assessment)

### Complete `app.py` for your Space:

```python
import torch
import numpy as np
import gradio as gr
from transformers import pipeline, VitsModel, VitsTokenizer

# Translation: any language -> English -> Dutch
asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-small")
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-nl")

# TTS: Dutch
tts_model = VitsModel.from_pretrained("facebook/mms-tts-nld")
tts_tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-nld")

def translate(audio):
    outputs = asr_pipe(audio, max_new_tokens=256, generate_kwargs={"task": "translate"})
    translated = translator(outputs["text"], max_length=512)
    return translated[0]["translation_text"]

def synthesise(text):
    inputs = tts_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = tts_model(inputs["input_ids"])
    return outputs.waveform[0].numpy()

target_dtype = np.int16
max_range = np.iinfo(target_dtype).max

def speech_to_speech_translation(audio):
    translated_text = translate(audio)
    synthesised_speech = synthesise(translated_text)
    synthesised_speech = (synthesised_speech * max_range).astype(np.int16)
    return 16000, synthesised_speech

demo = gr.Blocks()
mic_translate = gr.Interface(
    fn=speech_to_speech_translation,
    inputs=gr.Audio(sources="microphone", type="filepath"),
    outputs=gr.Audio(label="Generated Speech", type="numpy"),
)
file_translate = gr.Interface(
    fn=speech_to_speech_translation,
    inputs=gr.Audio(sources="upload", type="filepath"),
    outputs=gr.Audio(label="Generated Speech", type="numpy"),
)
with demo:
    gr.TabbedInterface([mic_translate, file_translate], ["Microphone", "Audio File"])

demo.launch()
```

## 7. Assessment Submission

Once your Space is deployed and working:

1. Go to: https://huggingface.co/spaces/huggingface-course/audio-course-u7-assessment
2. Enter your Space repository ID (e.g., `your-username/speech-to-speech-translation`)
3. The assessment will send a test audio to your demo and verify the output is non-English
4. Check your progress: https://huggingface.co/spaces/MariaK/Check-my-progress-Audio-Course